In [ ]:
##burst encoding with simulation time =1.
##this is a working code.
# Connect4 reference code:  https://github.com/neoyung/connect-4/tree/master
##DSQN reference code : https://github.com/mahmoudakl/dsrl
#wandb link: https://wandb.ai/kradeero-ohio-university/experiments/runs/rz0eegr6

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display
from itertools import count
import wandb
import os

# Initialize WandB
wandb.login(key='e21de1f4d4c13b4ba109db92ba20cc946e7da3c5')
wandb.init(project="experiments", name="burst_c4")

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Connect Four Environment
class ConnectX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1, 'draw': 0.5, 'lose': -1, 'step': -0.05}

    def render(self):
        rendered_board = self.board_state.astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = 'O'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state

        # Horizontal check
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id):
                    return True

        # Vertical check
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id):
                    return True

        # Diagonal checks
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]):
                    return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]):
                    return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['draw']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            print('Move is invalid')
            return self.board_state.copy(), self.reward['lose'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

    def check_potential_win(self, action, player):
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        if i < 0:  # Column is full
            return False
        original_value = self.board_state[i, action]
        self.board_state[i, action] = self.players[player]
        is_win = self.check_win(player)
        self.board_state[i, action] = original_value
        return is_win

# Replay Memory
class ReplayMemory:
    def __init__(self, capacity=100000):
        self.memory = []
        self.capacity = capacity
        self.position = 0

    def dump(self, transition):
        if len(self.memory) < self.capacity:
            self.memory.append(None)
        self.memory[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

# Surrogate Gradient Spike Function
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta

        torch.manual_seed(seed)

        # Initialize weights
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))

        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        mem = [torch.zeros(batch_size, self.weights[l].size(1), device=device) for l in range(len(self.weights))]
        spk = [[] for _ in range(len(self.weights))]
        mem_rec = []

        for t in range(self.simulation_time):
            input_t = x[:, t, :]
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.matmul(input_t, self.weights[l])
                else:
                    h = torch.matmul(spk[l-1][-1], self.weights[l])

                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])

        q_values = mem_rec[-1]
        return q_values, mem_rec, spk

# DSQN Agent
class DSQN:
    def __init__(self, discount_factor, dsnn_config):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.update_target_network()
        
        self.board_size = 6 * 7
        self.max_spikes_per_burst = 3  # Maximum spikes in a burst (for empty, value 0)
        self.burst_interval = 1  # Time steps between spikes in a burst
        self.input_size = self.board_size

    def burst_encode(self, states):
        batch_size = states.size(0)
        # Adjust encoded tensor for simulation_time=1
        encoded = torch.zeros(batch_size, self.simulation_time, self.board_size, device=device)
        
        # Flatten states to [batch_size, board_size]
        states_flat = states.view(batch_size, -1).long()
        
        # Map state values to intensity: 0 (empty) -> 3, 1 (O) -> 2, 2 (X) -> 1
        intensity = torch.where(states_flat == 0, 3.0, torch.where(states_flat == 1, 2.0, 1.0))
        
        # Place intensity in the single time step (t=0)
        encoded[:, 0, :] = intensity
        
        # Add small noise to simulate jitter
        noise = torch.normal(mean=0.0, std=0.01, size=encoded.shape, device=device)
        encoded = (encoded + noise).clamp(0, 3.0)  # Clamp to max intensity of empty
        
        return encoded

    def select_action(self, state, available_actions, training=True, steps_done=None):
        state = torch.tensor(state, dtype=torch.float, device=device).unsqueeze(0)
        encoded_state = self.burst_encode(state)
        
        if training:
            if steps_done is None:
                raise ValueError("steps_done is required when training=True")
            eps_threshold = EPS_END + (EPS_START - EPS_END) * np.exp(-steps_done / EPS_DECAY)
        else:
            eps_threshold = 0
        
        if random.random() > eps_threshold:
            with torch.no_grad():
                q_values, _, spk = self.training_net(encoded_state)
                # Sum spikes across hidden layers for the last timestep
                step_spike_count = sum(layer_spikes[-1].sum().item() for layer_spikes in spk[:-1] if layer_spikes)
                total_spike_count[0] += step_spike_count  # Global accumulator for spikes per step
                episode_count_for_avg_spikes[0] += 1
                q_values = q_values.flatten()
                valid_q = q_values[available_actions]
                return available_actions[torch.argmax(valid_q).item()]
        return random.choice(available_actions)

    def optimize_model(self, memory):
        if len(memory) < self.batch_size:
            return 0.0, 0
        
        transitions = memory.sample(self.batch_size)
        state_batch, action_batch, reward_batch, next_state_batch = zip(*transitions)
        
        state_batch = torch.tensor(np.array(state_batch), dtype=torch.float, device=device)
        action_batch = torch.tensor(action_batch, dtype=torch.long, device=device)
        reward_batch = torch.tensor(reward_batch, dtype=torch.float, device=device)
        non_final_mask = torch.tensor([s is not None for s in next_state_batch], device=device)
        non_final_next_states = [s for s in next_state_batch if s is not None]
        non_final_next_states = torch.tensor(np.array(non_final_next_states), dtype=torch.float, device=device) if non_final_next_states else None
        
        state_batch_encoded = self.burst_encode(state_batch)
        current_q, _, spk = self.training_net(state_batch_encoded)
        
        valid_action_mask = action_batch != -1
        current_q_selected = torch.zeros(self.batch_size, device=device)
        
        if valid_action_mask.any():
            valid_indices = valid_action_mask.nonzero(as_tuple=True)[0]
            valid_actions = action_batch[valid_action_mask].unsqueeze(1)
            current_q_selected[valid_indices] = current_q[valid_indices].gather(1, valid_actions).squeeze(1)
        
        next_state_values = torch.zeros(self.batch_size, device=device)
        if non_final_next_states is not None:
            non_final_next_states_encoded = self.burst_encode(non_final_next_states)
            with torch.no_grad():
                next_q_values, _, _ = self.target_net(non_final_next_states_encoded)
                next_state_values[non_final_mask] = next_q_values.max(1)[0]
        
        expected_q = reward_batch + (self.gamma * next_state_values)
        loss = F.smooth_l1_loss(current_q_selected[valid_action_mask], expected_q[valid_action_mask])
        
        self.training_net.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.training_net.parameters(), max_norm=1.0)
        self.training_net.optimizer.step()

        total_spikes = sum(sum(spk[l][-1].sum().item() for l in range(len(spk)-1) if spk[l]) for _ in range(self.batch_size))
        return loss.item(), total_spikes

    def update_target_network(self):
        self.target_net.load_state_dict(self.training_net.state_dict())

    def save_model(self, filename):
        """Save model weights to file"""
        torch.save({
            'training_net': self.training_net.state_dict(),
            'target_net': self.target_net.state_dict()
        }, filename)

    def load_model(self, filename):
        """Load model weights from file"""
        checkpoint = torch.load(filename)
        self.training_net.load_state_dict(checkpoint['training_net'])
        self.target_net.load_state_dict(checkpoint['target_net'])

# Random Agent
def random_agent(actions):
    return random.choice(actions)

# Evaluation Function
def evaluate_policy(agent, env, episodes=100):
    wins, losses, draws, moves_taken = 0, 0, 0, []
    for _ in range(episodes):
        state = env.reset()
        move_count = 0
        while not env.isDone:
            available_actions = env.get_available_actions()
            action = agent.select_action(state, available_actions, training=False)
            state, reward, valid = env.make_move(action, 'p1')
            move_count += 1
            if not valid:
                losses += 1
                break
            if env.isDone:
                if reward == 1:
                    wins += 1
                    moves_taken.append(move_count)
                elif reward == 0.5:
                    draws += 1
                break
            available_actions = env.get_available_actions()
            action = random_agent(available_actions)
            state, reward, _ = env.make_move(action, 'p2')
            move_count += 1
            if env.isDone:
                if reward == 1:
                    losses += 1
                elif reward == 0.5:
                    draws += 1
                break
    return wins / episodes, losses / episodes, draws / episodes, np.mean(moves_taken) if moves_taken else 0

# Hyperparameters
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TARGET_UPDATE = 10
MEMORY_CAPACITY = 100000
NUM_EPISODES = 20000
LOG_INTERVAL = 100

# DSNN configuration
dsnn_config = {
    'architecture': [6 * 7, 128, 128, 7],  # Input size for burst coding
    'seed': 42,
    'alpha': 0.9,
    'beta': 0.8,
    'weight_scale': 1.0,
    'batch_size': BATCH_SIZE,
    'threshold': 0.1,
    'simulation_time': 1,  
    'learning_rate': 0.0005,
    'reset_potential': 0.0
}

# Initialize environment and agent
env = ConnectX()
agent = DSQN(discount_factor=GAMMA, dsnn_config=dsnn_config)
memory = ReplayMemory()

# Global variables for spike tracking
total_spike_count = [0]
episode_count_for_avg_spikes = [0]

# Training loop
steps_done = 0
total_loss = 0.0
episode_count_for_avg_loss = 0
interval_o_wins = 0
interval_x_wins = 0
interval_draws = 0
interval_total_moves = 0

for episode in range(NUM_EPISODES):
    state = env.reset()
    move_count = 0
    outcome = None

    while True:
        available_actions = env.get_available_actions()
        action = agent.select_action(state, available_actions, steps_done=steps_done, training=True)
        steps_done += 1
        next_state, reward, valid = env.make_move(action, 'p1')
        move_count += 1
        if not valid:
            outcome = 'loss'
            memory.dump((state, action, reward, None))
            break
        if env.isDone:
            if reward == 1:
                outcome = 'win'
            elif reward == 0.5:
                outcome = 'draw'
            memory.dump((state, action, reward, None))
            break
        available_actions = env.get_available_actions()
        action_p2 = random_agent(available_actions)
        next_next_state, reward_p2, _ = env.make_move(action_p2, 'p2')
        move_count += 1
        reward = -reward_p2 if reward_p2 in [1, 0.5] else env.reward['step']
        memory.dump((state, action, reward, next_next_state))
        state = next_next_state
        loss, _ = agent.optimize_model(memory)
        total_loss += loss if loss else 0
        episode_count_for_avg_loss += 1 if loss else 0
        if env.isDone:
            if reward_p2 == 1:
                outcome = 'loss'
            elif reward_p2 == 0.5:
                outcome = 'draw'
            break

    if outcome == 'win':
        interval_o_wins += 1
    elif outcome == 'loss':
        interval_x_wins += 1
    elif outcome == 'draw':
        interval_draws += 1
    interval_total_moves += move_count

    if episode % TARGET_UPDATE == 0:
        agent.update_target_network()

    if (episode + 1) % LOG_INTERVAL == 0 or episode == NUM_EPISODES - 1:
        total_games_in_interval = episode % LOG_INTERVAL + 1 if episode == NUM_EPISODES - 1 and (episode + 1) % LOG_INTERVAL != 0 else LOG_INTERVAL
        if total_games_in_interval > 0 and interval_total_moves > 0:
            o_win_rate = interval_o_wins / total_games_in_interval
            o_loss_rate = interval_x_wins / total_games_in_interval
            o_draw_rate = interval_draws / total_games_in_interval
            o_win_draw_rate = o_win_rate + o_draw_rate
            interval_avg_moves = interval_total_moves / total_games_in_interval
            average_loss = total_loss / episode_count_for_avg_loss if episode_count_for_avg_loss > 0 else 0
            average_spikes = total_spike_count[0] / episode_count_for_avg_spikes[0] if episode_count_for_avg_spikes[0] > 0 else 0

            print(f"\n--- Episode {episode + 1}/{NUM_EPISODES} ---")
            print(f"Training Summary (last {total_games_in_interval} episodes, 'O' perspective):")
            print(f" 'O' Wins: {interval_o_wins}, Losses: {interval_x_wins}, Draws: {interval_draws}")
            print(f" Win Rate: {o_win_rate:.3f}, Loss Rate: {o_loss_rate:.3f}, Draw Rate: {o_draw_rate:.3f}, Win+Draw Rate: {o_win_draw_rate:.3f}")
            print(f" Avg Moves per game: {interval_avg_moves:.1f}")
            print(f" Average Loss per Episode: {average_loss:.6f}")
            print(f" Average Spikes per Move: {average_spikes:.2f}")

            wandb.log({
                "Episode": episode + 1,
                "Wins": interval_o_wins,
                "Losses": interval_x_wins,
                "Draws": interval_draws,
                "Win Rate": o_win_rate,
                "Loss Rate": o_loss_rate,
                "Draw Rate": o_draw_rate,
                "Win+Draw Rate": o_win_draw_rate,
                "Interval Avg Moves": interval_avg_moves,
                "Average Loss per Episode": average_loss,
                "Average Spikes per Move": average_spikes,
            })

        # Reset interval counters
        interval_o_wins = 0
        interval_x_wins = 0
        interval_draws = 0
        interval_total_moves = 0
        total_loss = 0.0
        episode_count_for_avg_loss = 0
        total_spike_count[0] = 0
        episode_count_for_avg_spikes[0] = 0

wandb.finish()
print("Training complete")
model_save_path = "../saved_models/c4_burst.pth"
# Save final model
torch.save({
    'training_net': agent.training_net.state_dict(),
    'target_net': agent.target_net.state_dict()
}, model_save_path)
print("Trained model saved to '../saved_models/c4_burst.pth'")

# Demo function
def demo():
    env.reset()
    env.render()
    while not env.isDone:
        state = env.board_state.copy()
        available_actions = env.get_available_actions()
        action = agent.select_action(state, available_actions, training=False)
        state, reward, valid = env.make_move(action, 'p1')
        env.render()
        if not valid:
            print("Invalid move by 'O'!")
            break
        if reward == 1:
            print("O Wins!")
            break
        if reward == 0.5:
            print("Draw!")
            break
        available_actions = env.get_available_actions()
        action = random_agent(available_actions)
        state, reward, _ = env.make_move(action, 'p2')
        env.render()
        if reward == 1:
            print("X Wins!")
            break
        elif reward == 0.5:
            print("Draw!")
            break

demo()

In [1]:

##Evaluation Code for burst encoding
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from IPython.display import display

# Device setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Connect Four Environment
class ConnectFourX:
    def __init__(self, height=6, width=7):
        self.board_height = height
        self.board_width = width
        self.board_state = np.zeros((height, width), dtype=np.int8)
        self.players = {'p1': 1, 'p2': 2}
        self.isDone = False
        self.reward = {'win': 1.0, 'r': 0.5, 'b': -1.0, 'step': 0.0}

    def render(self, visualize=False):
        if not visualize:
            return
        rendered_board = self.board_state.copy().astype(str)
        rendered_board[self.board_state == 0] = ' '
        rendered_board[self.board_state == 1] = '0'
        rendered_board[self.board_state == 2] = 'X'
        display(pd.DataFrame(rendered_board))

    def reset(self):
        self.board_state = np.zeros((self.board_height, self.board_width), dtype=np.int8)
        self.isDone = False
        return self.board_state.copy()

    def get_available_actions(self):
        return [j for j in range(self.board_width) if self.board_state[0, j] == 0]

    def check_win(self, player):
        player_id = self.players[player]
        board = self.board_state
        # Horizontal check
        for i in range(self.board_height):
            for j in range(self.board_width - 3):
                if np.all(board[i, j:j+4] == player_id):
                    return True
        # Vertical check
        for j in range(self.board_width):
            for i in range(self.board_height - 3):
                if np.all(board[i:i+4, j] == player_id):
                    return True
        # Diagonal checks
        for i in range(self.board_height - 3):
            for j in range(self.board_width - 3):
                if np.all([board[i+k, j+k] == player_id for k in range(4)]):
                    return True
            for j in range(3, self.board_width):
                if np.all([board[i+k, j-k] == player_id for k in range(4)]):
                    return True
        return False

    def check_game_done(self, player):
        if self.check_win(player):
            self.isDone = True
            return self.reward['win']
        if not np.any(self.board_state == 0):
            self.isDone = True
            return self.reward['r']
        return self.reward['step']

    def make_move(self, action, player):
        if action not in self.get_available_actions():
            return self.board_state.copy(), self.reward['b'], False
        i = self.board_height - 1 - np.sum(self.board_state[:, action] != 0)
        self.board_state[i, action] = self.players[player]
        reward = self.check_game_done(player)
        return self.board_state.copy(), reward, True

# Surrogate Gradient Spike Function
class SurrGradSpike(torch.autograd.Function):
    scale = 100.0

    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        out = torch.zeros_like(input)
        out[input > 0] = 1.0
        return out

    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad = grad_input / (SurrGradSpike.scale * torch.abs(input) + 1.0) ** 2
        return grad

# Deep Spiking Neural Network (DSNN)
class DSNN(nn.Module):
    def __init__(self, architecture, seed, alpha, beta, weight_scale, batch_size, 
                 threshold, simulation_time, learning_rate, reset_potential=0):
        super().__init__()
        self.architecture = architecture
        self.simulation_time = simulation_time
        self.batch_size = batch_size
        self.threshold = threshold
        self.reset_potential = reset_potential
        self.alpha = alpha
        self.beta = beta
        
        torch.manual_seed(seed)
        
        self.weights = nn.ParameterList()
        for i in range(len(architecture)-1):
            w = torch.Tensor(architecture[i], architecture[i+1])
            nn.init.normal_(w, mean=0.0, std=weight_scale/np.sqrt(architecture[i]))
            self.weights.append(nn.Parameter(w))
            
        self.spike_fn = SurrGradSpike.apply
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def forward(self, x):
        batch_size = x.size(0)
        syn, mem, spk = [], [], []
        
        for l in range(len(self.weights)):
            syn.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            mem.append(torch.zeros(batch_size, self.weights[l].size(1), device=device))
            spk.append([])
        
        mem_rec = []
        all_spikes = []
        ac_count = 0
        internal_mac_count = 0
        
        for t in range(self.simulation_time):
            input = x[:, t, :]
            layer_spikes = []
            for l in range(len(self.weights)):
                if l == 0:
                    h = torch.mm(input, self.weights[l])
                    num_spikes = torch.sum(input > 0).item()
                else:
                    h = torch.mm(spk[l-1][-1], self.weights[l])
                    num_spikes = torch.sum(spk[l-1][-1]).item()
                ac_count += num_spikes * self.weights[l].size(1)
                
                num_neurons = self.weights[l].size(1)
                internal_mac_count += num_neurons * 2
                
                syn[l] = self.alpha * syn[l] + h
                mem[l] = self.beta * mem[l] + syn[l]
                
                if l < len(self.weights)-1:
                    mthr = mem[l] - self.threshold
                    spk_current = self.spike_fn(mthr)
                    internal_mac_count += num_neurons * 2 * torch.mean(spk_current).item()
                    mem[l] = mem[l] * (1 - spk_current) + self.reset_potential * spk_current
                    spk[l].append(spk_current)
                    layer_spikes.append(spk_current)
                else:
                    layer_spikes.append(torch.zeros_like(mem[l]))
                
                if l == len(self.weights)-1:
                    mem_rec.append(mem[l])
                
            all_spikes.append(layer_spikes)
        
        q_values = mem[-1]
        return q_values, mem_rec, all_spikes, ac_count, internal_mac_count

# DSQN Agent with Burst Encoding
class DSQN:
    def __init__(self, discount_factor=0.99, dsnn_config=None):
        self.gamma = discount_factor
        self.batch_size = dsnn_config['batch_size']
        self.simulation_time = dsnn_config['simulation_time']
        self.training_net = DSNN(**dsnn_config).to(device)
        self.target_net = DSNN(**dsnn_config).to(device)
        self.board_size = 6 * 7
        self.max_spikes_per_burst = 3
        self.burst_interval = 1

    def burst_encode(self, states):
        batch_size = states.size(0)
        encoded = torch.zeros(batch_size, self.simulation_time, self.board_size, device=device)
        states_flat = states.view(batch_size, -1).long()
        intensity = torch.where(states_flat == 0, 3.0, torch.where(states_flat == 1, 2.0, 1.0))
        encoded[:, 0, :] = intensity
        noise = torch.normal(mean=0.0, std=0.01, size=encoded.shape, device=device)
        encoded = (encoded + noise).clamp(0, 3.0)
        return encoded

    def observe(self, state, action_space=None):
        with torch.no_grad():
            state_tensor = torch.tensor(state, dtype=torch.float, device=device).unsqueeze(0)
            encoded_state = self.burst_encode(state_tensor)
            q_values, mem_rec, all_spikes, ac_count, internal_mac_count = self.training_net(encoded_state)
            q_values = q_values.cpu().numpy().flatten()
            if action_space is not None:
                valid_q = [(q_values[a], a) for a in action_space]
                action = max(valid_q, key=lambda x: x[0])[1]
            else:
                action = np.argmax(q_values)
            return action, mem_rec, all_spikes, ac_count, internal_mac_count

    def load_model(self, filename):
        try:
            checkpoint = torch.load(filename, map_location=device)
            self.training_net.load_state_dict(checkpoint['training_net'])
            self.target_net.load_state_dict(checkpoint['target_net'])
            self.training_net.eval()
            self.target_net.eval()
            print(f"DSQN model loaded from {filename}")
        except FileNotFoundError:
            raise FileNotFoundError(f"DSQN model file {filename} not found")

# Analysis Functions
def analyze_decision_stabilization(mem_rec, simulation_time):
    decision_times = []
    final_decision = torch.argmax(mem_rec[-1], dim=1)
    for b in range(mem_rec[0].size(0)):
        stabilized = False
        for t in range(simulation_time):
            current_decision = torch.argmax(mem_rec[t], dim=1)[b]
            if current_decision == final_decision[b]:
                stable = True
                for t_next in range(t, simulation_time):
                    if torch.argmax(mem_rec[t_next], dim=1)[b] != final_decision[b]:
                        stable = False
                        break
                if stable:
                    decision_times.append(t + 1)
                    stabilized = True
                    break
        if not stabilized:
            decision_times.append(simulation_time)
    return decision_times

def analyze_spikes(all_spikes, architecture, ac_count, internal_mac_count):
    total_spikes = 0
    total_neurons = sum(architecture[1:-1])
    spike_counts = []
    
    for t in range(len(all_spikes)):
        for l in range(len(all_spikes[t])):
            if l < len(architecture) - 1:
                spikes = all_spikes[t][l]
                spike_count = torch.sum(spikes).item()
                total_spikes += spike_count
                spike_counts.append(spike_count)
    
    sparsity = 1 - (total_spikes / (total_neurons * len(all_spikes)))
    return total_spikes, sparsity, spike_counts, ac_count, internal_mac_count

# Utility Functions
def random_move(env):
    return random.choice(env.get_available_actions())

def play_game(agent, env, agent_first=True, visualize=False, collect_analysis=False):
    state = env.reset()
    current_player = 'p1' if agent_first else 'p2'
    agent_symbol = '0' if agent_first else 'X'
    random_symbol = 'X' if agent_first else '0'

    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'internal_mac_counts': []
    } if collect_analysis else None

    if visualize:
        print("\n=== New Game ===")
        print(f"Agent plays as {agent_symbol}, Random plays as {random_symbol}")
        env.render(visualize=True)

    while not env.isDone:
        if current_player == 'p1':
            if agent_first:
                action_space = env.get_available_actions()
                if collect_analysis:
                    action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(state, action_space)
                    total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(all_spikes, agent.training_net.architecture, ac_count, internal_mac_count)
                    analysis_data['total_spikes'].append(total_spikes)
                    analysis_data['sparsity'].append(sparsity)
                    analysis_data['ac_counts'].append(ac_count)
                    analysis_data['internal_mac_counts'].append(internal_mac_count)
                    decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
                    analysis_data['decision_times'].extend(decision_times)
                else:
                    action = agent.observe(state, action_space)[0]
                next_state, reward, valid = env.make_move(action, 'p1')
                if not valid:
                    if visualize:
                        print(f"Invalid move by Agent ({agent_symbol})!")
                    return 'random', analysis_data
                state = next_state
                if visualize:
                    print(f"Agent ({agent_symbol}) played at column {action}")
                    env.render(visualize=True)
            else:
                action = random_move(env)
                next_state, reward, _ = env.make_move(action, 'p1')
                state = next_state
                if visualize:
                    print(f"Random ({random_symbol}) played at column {action}")
                    env.render(visualize=True)
        else:
            if agent_first:
                action = random_move(env)
                next_state, reward, _ = env.make_move(action, 'p2')
                state = next_state
                if visualize:
                    print(f"Random ({random_symbol}) played at column {action}")
                    env.render(visualize=True)
            else:
                action_space = env.get_available_actions()
                if collect_analysis:
                    action, mem_rec, all_spikes, ac_count, internal_mac_count = agent.observe(state, action_space)
                    total_spikes, sparsity, _, ac_count, internal_mac_count = analyze_spikes(all_spikes, agent.training_net.architecture, ac_count, internal_mac_count)
                    analysis_data['total_spikes'].append(total_spikes)
                    analysis_data['sparsity'].append(sparsity)
                    analysis_data['ac_counts'].append(ac_count)
                    analysis_data['internal_mac_counts'].append(internal_mac_count)
                    decision_times = analyze_decision_stabilization(mem_rec, agent.simulation_time)
                    analysis_data['decision_times'].extend(decision_times)
                else:
                    action = agent.observe(state, action_space)[0]
                next_state, reward, valid = env.make_move(action, 'p2')
                if not valid:
                    if visualize:
                        print(f"Invalid move by Agent ({agent_symbol})!")
                    return 'random', analysis_data
                state = next_state
                if visualize:
                    print(f"Agent ({agent_symbol}) played at column {action}")
                    env.render(visualize=True)

        if env.isDone:
            if visualize:
                if reward == 0.5:
                    print("Game ended in a draw!")
                elif (agent_first and reward == 1 and current_player == 'p1') or (not agent_first and reward == 1 and current_player == 'p2'):
                    print(f"Agent ({agent_symbol}) wins!")
                else:
                    print(f"Random ({random_symbol}) wins!")
            if reward == 0.5:
                return 'r', analysis_data
            elif (agent_first and reward == 1 and current_player == 'p1') or (not agent_first and reward == 1 and current_player == 'p2'):
                return 'agent', analysis_data
            else:
                return 'random', analysis_data

        current_player = 'p2' if current_player == 'p1' else 'p1'

def test_agent_vs_random(agent, agent_name, env, num_games=100, visualize_all=False):
    results_agent_first = {'agent': 0, 'random': 0, 'r': 0}
    analysis_data = {
        'total_spikes': [],
        'sparsity': [],
        'decision_times': [],
        'ac_counts': [],
        'internal_mac_counts': []
    }

    # Agent goes first ('0')
    agent.load_model("../saved_models/c4_burst.pth")
    for i in range(num_games):
        visualize = visualize_all
        winner, game_analysis = play_game(agent, env, agent_first=True, visualize=visualize, collect_analysis=True)
        results_agent_first[winner] += 1
        if game_analysis:
            analysis_data['total_spikes'].extend(game_analysis['total_spikes'])
            analysis_data['sparsity'].extend(game_analysis['sparsity'])
            analysis_data['decision_times'].extend(game_analysis['decision_times'])
            analysis_data['ac_counts'].extend(game_analysis['ac_counts'])
            analysis_data['internal_mac_counts'].extend(game_analysis['internal_mac_counts'])

    # Print game results
    print(f"\n{agent_name} vs. Random Agent - Final Results")
    print("="*50)
    print(f"{agent_name} goes first ({agent_name} as 0, Random as X):")
    print(f"Wins ({agent_name}): {results_agent_first['agent']}, Losses: {results_agent_first['random']}, Draws: {results_agent_first['r']}")
    print(f"Win rate: {results_agent_first['agent'] / num_games:.2%}, "
          f"Loss rate: {results_agent_first['random'] / num_games:.2%}, "
          f"Draw rate: {results_agent_first['r'] / num_games:.2%}")

    # Print analysis results
    if analysis_data['total_spikes']:
        print(f"\nDSQN (Burst Encoding) Analysis Results")
        print("="*50)
        print(f"Average Total Spikes per Decision: {float(np.mean(analysis_data['total_spikes'])):.2f} "
              f"(Std: {float(np.std(analysis_data['total_spikes'])):.2f})")
        print(f"Average Sparsity per Decision: {float(np.mean(analysis_data['sparsity'])):.2%} "
              f"(Std: {float(np.std(analysis_data['sparsity'])):.2%})")
        print(f"Average ACs per Decision (spike-triggered additions): {float(np.mean(analysis_data['ac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['ac_counts'])):.2f})")
        print(f"Average Internal State Update MACs per Decision: {float(np.mean(analysis_data['internal_mac_counts'])):.2f} "
              f"(Std: {float(np.std(analysis_data['internal_mac_counts'])):.2f})")
        decision_time_counts = np.bincount(analysis_data['decision_times'], minlength=agent.simulation_time + 1)[1:]
        print(f"Decision Stabilization Times (over all decisions):")
        for t in range(agent.simulation_time):
            print(f"  Time Step {t+1}: {decision_time_counts[t]} decisions "
                  f"({decision_time_counts[t] / len(analysis_data['decision_times']):.2%})")
        # Compute energy ratio and savings
        total_spikes_avg = float(np.mean(analysis_data['total_spikes']))
        total_neurons = sum(agent.training_net.architecture[1:-1])
        total_possible_spikes = total_neurons * agent.simulation_time
        f_r = total_spikes_avg / total_possible_spikes
        avg_ac = float(np.mean(analysis_data['ac_counts']))
        avg_internal_mac = float(np.mean(analysis_data['internal_mac_counts']))
        ann_energy = 22656 * 31
        snn_energy = avg_ac + avg_internal_mac * 31
        energy_ratio_detailed = snn_energy / ann_energy
        energy_savings_detailed = (1 - energy_ratio_detailed) * 100
        energy_ratio_simplified = agent.simulation_time * f_r * (1 / 31)
        energy_savings_simplified = (1 - energy_ratio_simplified) * 100
        print(f"Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): {energy_ratio_simplified:.4f}")
        print(f"Simplified Energy Savings: {energy_savings_simplified:.1f}%")
        print(f"Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): {energy_ratio_detailed:.4f}")
        print(f"Detailed Energy Savings: {energy_savings_detailed:.1f}%")

if __name__ == "__main__":
    # Set random seeds for reproducibility
    random.seed(82)
    np.random.seed(82)
    torch.manual_seed(82)

    # DSNN configuration
    dsnn_config = {
        'architecture': [6 * 7, 128, 128, 7],
        'seed': 82,
        'alpha': 0.9,
        'beta': 0.8,
        'weight_scale': 1.0,
        'batch_size': 128,
        'threshold': 0.1,
        'simulation_time': 5,
        'learning_rate': 0.0005,
        'reset_potential': 0.0
    }

    # Initialize environment and agent
    env = ConnectFourX()
    dsqn_agent = DSQN(discount_factor=0.99, dsnn_config=dsnn_config)

    print("\nTesting DSQN (Burst Encoding) vs. Random Agent")
    test_agent_vs_random(dsqn_agent, "DSQN", env, num_games=100, visualize_all=False)


Device: cpu

Testing DSQN (Burst Encoding) vs. Random Agent
DSQN model loaded from ../saved_models/c4_burst.pth

DSQN vs. Random Agent - Final Results
DSQN goes first (DSQN as 0, Random as X):
Wins (DSQN): 94, Losses: 6, Draws: 0
Win rate: 94.00%, Loss rate: 6.00%, Draw rate: 0.00%

DSQN (Burst Encoding) Analysis Results
Average Total Spikes per Decision: 579.99 (Std: 0.48)
Average Sparsity per Decision: 54.69% (Std: 0.04%)
Average ACs per Decision (spike-triggered additions): 64989.10 (Std: 873.19)
Average Internal State Update MACs per Decision: 3789.98 (Std: 0.96)
Decision Stabilization Times (over all decisions):
  Time Step 1: 498 decisions (100.00%)
  Time Step 2: 0 decisions (0.00%)
  Time Step 3: 0 decisions (0.00%)
  Time Step 4: 0 decisions (0.00%)
  Time Step 5: 0 decisions (0.00%)
Simplified SNN/ANN Energy Ratio (T * f_r * E_AC/E_MAC): 0.0731
Simplified Energy Savings: 92.7%
Detailed SNN/ANN Energy Ratio (ACs + Internal MACs): 0.2598
Detailed Energy Savings: 74.0%
